# Rent 05 · Clean HPD complaints
**Input:** `data/raw/hpd_complaints/` (one row per problem, Jan 2022 → now)
**Output:** `data/processed/hpd_complaints_clean.parquet`

Steps, all in `src/clean_housing.py`:
1. Drop duplicate `problem_id`s.
2. Parse dates, keeping 2022+.
3. Keep rows with a valid 10-digit BBL, the key that links every dataset.
4. Map HPD's 13 categories onto **renter-friendly problem types**, the same vocabulary we use for violations.
5. Add flags: emergency, duplicate report, building-wide (no apartment), still open.

In [1]:
import sys, pathlib, json
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT))

import pandas as pd
from src import config, housing as H, clean_housing as C
from src.preprocess import CleaningReport

pd.set_option("display.float_format", "{:,.2f}".format)
pd.set_option("display.max_colwidth", 70)

def show_report(rep):
    return pd.DataFrame(rep.steps)[["step", "rows_before", "rows_after", "rows_removed", "detail"]]

def save(df, name, rep):
    df.to_parquet(C.OUT[name], index=False)
    (config.REPORTS_DIR / f"cleaning_report_{name}.json").write_text(json.dumps(rep.to_dict(), indent=2, default=str))
    print(f"Saved {len(df):,} rows × {df.shape[1]} columns → {C.OUT[name].relative_to(ROOT)}")

In [2]:
raw = H.load("hpd_complaints")
rep = CleaningReport("hpd_complaints")
clean = C.clean_complaints(raw, rep)
show_report(rep)

,step,rows_before,rows_after,rows_removed,detail
0,load_raw,4032908,4032908,0,"2,448,291 complaints"
1,drop_duplicate_problem_id,4032908,4032908,0,
2,parse_dates,4032908,4032908,0,missing or pre-2022 received_date
3,valid_bbl,4032908,4019416,13492,missing or invalid BBL
4,derive_fields,4019416,4019416,0,"problem_type, emergency, duplicate, building-wide, open flags"


### How HPD categories map onto our problem types

In [3]:
pd.crosstab(clean["major_category"], clean["problem_type"]).loc[lambda d: d.sum(axis=1).sort_values(ascending=False).index]

problem_type,doors/windows,electric,heat/hot water,leaks/mold,other,paint/plaster,pests,plumbing,safety,sanitation
major_category,,,,,,,,,,
HEAT/HOT WATER,0,0,1358045,0,0,0,0,0,0,0
UNSANITARY CONDITION,0,0,0,191511,0,0,360258,0,0,114010
PLUMBING,0,0,0,0,0,0,0,385863,0,0
PAINT/PLASTER,0,0,0,0,0,353184,0,0,0,0
DOOR/WINDOW,275608,0,0,0,0,0,0,0,0,0
WATER LEAK,0,0,0,223945,0,0,0,0,0,0
GENERAL,0,0,0,0,184877,0,0,0,0,0
ELECTRIC,0,164517,0,0,0,0,0,0,0,0
FLOORING/STAIRS,0,0,0,0,155310,0,0,0,0,0


In [4]:
summary = clean.groupby("problem_type").agg(problems=("problem_id", "size"), buildings=("bbl", "nunique"),
                                            emergency=("is_emergency", "mean"), still_open=("is_open", "mean"))
summary.sort_values("problems", ascending=False)

,problems,buildings,emergency,still_open
problem_type,,,,
heat/hot water,1358045,81252,1.00,0.00
other,490539,52287,0.24,0.02
leaks/mold,415456,54250,0.70,0.01
plumbing,385863,51511,0.74,0.02
pests,360258,48649,0.04,0.02
paint/plaster,353184,42767,0.31,0.01
doors/windows,275608,40207,0.78,0.02
electric,164517,38622,0.54,0.02
sanitation,114010,30177,0.15,0.02


In [5]:
print(f"Flagged duplicate reports: {clean['is_duplicate_report'].mean():.1%} (kept, flagged; models count distinct complaints)")
print(f"Building-wide problems: {clean['is_building_wide'].mean():.1%}")
save(clean, "hpd_complaints", rep)

Flagged duplicate reports: 15.9% (kept, flagged; models count distinct complaints)
Building-wide problems: 31.9%


Saved 4,019,416 rows × 15 columns → data/processed/hpd_complaints_clean.parquet
